## 7.1 Introduction to instruction fine-tuning

1) Dataset download and formatting

![example](https://raw.githubusercontent.com/ipdor/Pictures/master/20260507153028711.png)

让我们下载并格式化用于对预训练 LLM 进行指令微调的指令数据集。该数据集包含 1,100 个类似于图 7.2 中的指令-回复对。

In [1]:
import json
import os
import requests

In [2]:
def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with requests.get(url) as response:
            text_data = response.content.decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    # else:
    #     with open(file_path, "r", encoding="utf-8") as file:
    #         text_data = file.read()
    with open(file_path, "r") as file:
        data = json.load(file)
    return data

file_path = "instruction-data.json"
url= ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
"/main/ch07/01_main-chapter-code/instruction-data.json")

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


In [3]:
print("Example entry:\n", data[50])

Example entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


每条记录包含 `instruction`, `input` 和 `output`

In [4]:
print(data[2]["instruction"])

Convert 45 kilometers to meters.


In [5]:
# input 域有时候为空
print(data[2]["input"])

In [6]:
print(data[2]["output"])

45 kilometers is 45000 meters.


In [7]:
print("Another example entry:\n", data[999])

Another example entry:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


![two different ways of formatting](https://raw.githubusercontent.com/ipdor/Pictures/master/20260507171147257.png)

指令微调涉及在输入-输出对的数据集上训练模型，有不同的方法对记录进行格式化，图7.4是其中两种。

### 练习7.1 改变提示风格  

在使用Alpaca提示风格对模型进行微调后，尝试图7.4所示的Phi-3提示风格，并观察这是否会影响模型的响应质量。

In [8]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    # 如果数据的input部分为空，则输出直接跳过input域
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )

    return instruction_text + input_text

In [9]:
format_input(data[0])

'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nEvaluate the following phrase by transforming it into the spelling given.\n\n### Input:\nfreind --> friend'

In [10]:
model_input = format_input(data[50])
desired_response = f"\n\n## Response:\n{data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

## Response:
The correct spelling is 'Occasion.'


如果 `input` 域为空则会跳过

In [11]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


In [12]:
train_portion = int(len(data) * 0.85) # 85%作为训练数据
test_portion = int(len(data) * 0.1)
val_portion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion: train_portion+test_portion]
val_data = data[train_portion+test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


## 7.3 Organizing data into training batches

2) Batching the dataset

和之前微调分类器一样，需要填充每个样本到相同长度

![batching process](https://raw.githubusercontent.com/ipdor/Pictures/master/20260507180806809.png)

下面是 `InstructionDataset` 类的实现，对应图7.6的前两步 2.1 和 2.2。

2.1 组成完整Instruction + Response文本    
2.2 对每条完整文本编码成token ids序列

In [13]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        # 对数据预分词 Pretokenizes
        for entry in data: 
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            # 存放编码后的完整请求+回应的list
            self.encoded_texts.append(tokenizer.encode(full_text))
    
    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [14]:
import tiktoken

In [15]:
tokenizer = tiktoken.get_encoding("gpt2")
ind = InstructionDataset(data, tokenizer)
# print(ind[:2])
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


2.3 使用填充标记调整至相同长度。  

需要对每个batch进行填充，长度按照组内最长长度

In [42]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

# stack把多个同形状的 1D tensor 沿新维度拼接，变成一个 2D tensor。 注意输入必须是tensor
torch.stack([a, b])
# tensor([[1, 2, 3],
#         [4, 5, 6]])   shape: (2, 3)

tensor([[1, 2, 3],
        [4, 5, 6]])

In [ ]:
# 处理一个batch的token_id数据
def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):
    # 找到批次中最长的序列 
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst = []

    # 填充并准备输入
    for item in batch:
        new_item = item.copy()      # 避免对原始数据修改
        new_item += [pad_token_id]  # 额外填充标记，为下个版本做铺垫

        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1]) # 移除之前添加的额外填充标记
        inputs_lst.append(inputs)
    
    # 使用torrch.stack转换为 (batch_size, seq_len) 的标准矩阵格式
    # 将输入列表转换为张量并将其转移到目标设备
    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

In [17]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (inputs_1,
        inputs_2,
        inputs_3)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


2.4 为训练创建目标标记ID

除了每个 input_batch 之外，还需要对应的 target_batch，由输入批次右移一个token得到。最右侧的空位填充`<|endoffile|>`

In [ ]:
def custom_collate_draft_2(batch, pad_token_id=50256, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id] # 在 [data, pad_token_id] 的基础上填充，最终一定会多一个pad_token_id

        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1]) # 为输入batch去掉结尾多余token
        outputs = torch.tensor(padded[1:]) # 右移1得到输出batch
        # 不直接追加放入lst, 为后面做铺垫
        inputs_lst.append(inputs)
        targets_lst.append(outputs)
    
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [48]:
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


2.5 将填充令牌替换为占位符

![replace padding tokens with placeholders](https://raw.githubusercontent.com/ipdor/Pictures/master/20260507223256135.png)

除第一个填充标记外，其他所有替换为占位符`-100`

In [ ]:
targets = torch.tensor([    8,     9, 50256, 50256, 50256])
print(targets)
mask = targets == 50256 # 判断各位是否是 pad_token_id
print(mask)
print(torch.nonzero(mask))
indices = torch.nonzero(mask).squeeze() # 找出所有 True(是pad_token_id) 的位置索引
print(indices, indices.numel())
# 如果 pad_token_id 个数大于1， 那么把第一个 pad 之后的所有 pad 设为 ignore_index：
if indices.numel() > 1:
    targets[indices[1:]] = -100
print(targets)

tensor([    8,     9, 50256, 50256, 50256])
tensor([False, False,  True,  True,  True])
tensor([[2],
        [3],
        [4]])
tensor([2, 3, 4]) 3
tensor([    8,     9, 50256,  -100,  -100])


In [83]:
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        # 用 ignore_index 替换所有不是第一个的padding tokens
        mask = targets == pad_token_id
        indices = mask.nonzero().squeeze()
        if indices.numel() >1:
            targets[indices[1:]] = ignore_index

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [84]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


替换为 -100 的原因是计算 `cross entropy` 时刚好可以被忽略。

In [88]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],
    [-0.5, 1.5]]
)
targets_1 = torch.tensor([0, 1]) # Correct token indices to generate
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

tensor(1.1269)


In [89]:
logits_2 = torch.tensor(
[[-1.0, 1.0],
[-0.5, 1.5],
[-0.5, 1.5]]
)
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(0.7936)


In [ ]:
# 和2个token的 logits_1 的输出相等
targets_3 = torch.tensor([0, 1, -100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
loss_1 == loss_3: tensor(True)


In [ ]:
targets_3 = torch.tensor([0, 1, 0]) # 替换成非0/1/-100的数值会报错
loss_4 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_4)

tensor(1.4603)


### 练习 7.2 指令和输入屏蔽 

完成本章内容并使用 InstructionDataset 对模型进行微调后，用 -100 掩码替换指令和输入 token，以使用如图 7.13 所示的指令屏蔽方法。然后评估这对模型性能是否产生了积极影响。

## 7.4 Creating data loaders for an instruction dataset

In [97]:
print(1)

1
